Next word prediction

## Next Word Prediction *italicized text*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Step 1: Prepare a simple dataset
corpus = [
    "Dhaval loves pizza",
    "Dhaval eats sushi",
    "Baby Yoda is cute",
    "Yoda eats pizza"
]

# Tokenize and build vocabulary
words = set(" ".join(corpus).lower().split())
word_to_idx = {word: idx for idx, word in enumerate(words)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Prepare sequences for training
data = []
for sentence in corpus:
    tokens = sentence.lower().split()
    for i in range(1, len(tokens)):
        input_seq = [word_to_idx[token] for token in tokens[:i]]
        target_word = word_to_idx[tokens[i]]
        data.append((input_seq, target_word))

# Pad sequences to the same length
max_len = max(len(seq) for seq, _ in data)
padded_data = [(seq + [0] * (max_len - len(seq)), target) for seq, target in data]

# Convert data to tensors
inputs = torch.tensor([seq for seq, _ in padded_data], dtype=torch.long)
targets = torch.tensor([target for _, target in padded_data], dtype=torch.long)

# Create a simple Dataset and DataLoader
class SimpleDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

dataloader = DataLoader(SimpleDataset(inputs, targets), batch_size=2, shuffle=True)

# Step 2: Define the RNN model
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        rnn_out, _ = self.rnn(x)
        output = self.fc(rnn_out[:, -1, :])  # Take the last time step
        return output

# Step 3: Train the model
vocab_size = len(word_to_idx)
model = SimpleRNN(vocab_size, embedding_dim=8, hidden_dim=16)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(20):  # Fewer epochs for simplicity
    for batch_inputs, batch_targets in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_inputs)
        loss = criterion(outputs, batch_targets)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

# Step 4: Predict the next word
def predict_next(model, input_words):
    model.eval()
    with torch.no_grad():
        input_indices = [word_to_idx.get(word, 0) for word in input_words]
        input_indices = input_indices + [0] * (max_len - len(input_indices))
        input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0)
        output = model(input_tensor)
        predicted_idx = torch.argmax(output, dim=1).item()
        return idx_to_word[predicted_idx]

# Test prediction
sequence = ["baby", "yoda","is"]
predicted_word = predict_next(model, sequence)
print(f"Next word prediction: {predicted_word}")


Epoch 1, Loss: 2.7971
Epoch 2, Loss: 1.9394
Epoch 3, Loss: 1.7019
Epoch 4, Loss: 1.9480
Epoch 5, Loss: 1.5635
Epoch 6, Loss: 2.0752
Epoch 7, Loss: 0.6372
Epoch 8, Loss: 1.2595
Epoch 9, Loss: 0.7420
Epoch 10, Loss: 0.7911
Epoch 11, Loss: 0.7191
Epoch 12, Loss: 0.8281
Epoch 13, Loss: 0.6511
Epoch 14, Loss: 0.5312
Epoch 15, Loss: 0.2721
Epoch 16, Loss: 0.2110
Epoch 17, Loss: 0.1770
Epoch 18, Loss: 0.2123
Epoch 19, Loss: 0.0783
Epoch 20, Loss: 0.1404
Next word prediction: cute


Next character prediction

In [ ]:
# Step 1: Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Step 2: Define text and character set
text = "This is whitebox a software training institute"
chars = sorted(list(set(text)))
char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

# Step 3: Create sequences and labels
seq_length = 3
sequences = []
labels = []

for i in range(len(text) - seq_length):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

X = np.array(sequences)
y = np.array(labels)

# Step 4: One-hot encode sequences and labels
X_one_hot = tf.one_hot(X, len(chars))
y_one_hot = tf.one_hot(y, len(chars))

# Step 5: Build the model
model = Sequential()
model.add(SimpleRNN(50, input_shape=(seq_length, len(chars)), activation='relu'))
model.add(Dense(len(chars), activation='softmax'))

# Step 6: Compile and train the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_one_hot, y_one_hot, epochs=100)

# Step 7: Generate text
start_seq = "This is w"
generated_text = start_seq

for i in range(50):
    x = np.array([[char_to_index[char] for char in generated_text[-seq_length:]]])
    x_one_hot = tf.one_hot(x, len(chars))
    prediction = model.predict(x_one_hot)
    next_index = np.argmax(prediction)
    next_char = index_to_char[next_index]
    generated_text += next_char

print("Generated Text:")
print(generated_text)


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.0778 - loss: 2.8928
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0673 - loss: 2.8655
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0828 - loss: 2.8457
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1243 - loss: 2.8249
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1710 - loss: 2.8014 
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1710 - loss: 2.7767 
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1710 - loss: 2.7703 
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2125 - loss: 2.7376 
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2125 - loss: 2.7220 
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1761 - loss: 2.7148 
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2229 - loss: 2.6910 
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2539 - los

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Define a small corpus
corpus = [
    "Dhaval loves pizza",
    "Dhaval eats sushi",
    "Baby Yoda is cute",
    "Yoda eats pizza"
]

# Step 2: Tokenize the sentences and build vocabulary
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word_to_index = tokenizer.word_index
index_to_word = {v: k for k, v in word_to_index.items()}

# Convert sentences to sequences of word indices
sequences = tokenizer.texts_to_sequences(corpus)

# Step 3: Prepare input sequences and labels
input_sequences = []
labels = []

for seq in sequences:
    for i in range(1, len(seq)):
        input_sequences.append(seq[:i])  # Take words up to index i
        labels.append(seq[i])           # Predict the next word at index i

# Pad sequences to make them the same length
max_len = max(len(seq) for seq in input_sequences)
X = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
y = np.array(labels)

# Step 4: One-hot encode labels
vocab_size = len(word_to_index) + 1
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# Step 5: Build the model
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=10, input_length=max_len))
model.add(SimpleRNN(50, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))

# Step 6: Compile and train the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y_one_hot, epochs=50, verbose=1)

# Step 7: Predict the next word
def predict_next_word(model, tokenizer, input_text, max_len):
    sequence = tokenizer.texts_to_sequences([input_text])[0]
    padded_sequence = pad_sequences([sequence], maxlen=max_len, padding='pre')
    prediction = model.predict(padded_sequence)
    predicted_word = index_to_word[np.argmax(prediction)]
    return predicted_word

# Test prediction
input_text = "Dhaval"
predicted_word = predict_next_word(model, tokenizer, input_text, max_len)
print(f"Next word prediction: {predicted_word}")


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2222 - loss: 2.2979
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.2222 - loss: 2.2912
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.3333 - loss: 2.2850
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.4444 - loss: 2.2788
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.4444 - loss: 2.2728
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5556 - loss: 2.2667
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5556 - loss: 2.2607
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5556 - loss: 2.2545
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5556 - loss: 2.2482
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5556 - loss: 2.2418
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5556 - loss: 2.2352
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.4444 - loss: 2.2284
E